In [0]:
PATH_ACTIVITY_FILES = "/Volumes/customer_360/raw/source_files/landing_data/activities/"
PATH_ACTIVITY_CHECKPOINTLOCATION_BRONZE = "/Volumes/customer_360/raw/source_files/checkpoints/activities/"
TABLE_BRONZE_ACTIVITY = "customer_360.bronze.activities"
TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS customer_360.bronze.activities (
    activity_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    activity_type STRING,
    activity_channel STRING,
    activity_time TIMESTAMP,
    description STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

activity_schema = StructType([
    StructField("activity_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("activity_type", StringType(), True),
    StructField("activity_channel", StringType(), True),
    StructField("activity_time", TimestampType(), True),
    StructField("description", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])

In [0]:
activity_bronze=(
    spark
    .readStream
    .format("csv")
    .option("header",True)
    .schema(activity_schema)
    .load(PATH_ACTIVITY_FILES)
)

In [0]:

query=(
    activity_bronze
    .writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option("checkpointlocation",PATH_ACTIVITY_CHECKPOINTLOCATION_BRONZE)
    .toTable(TABLE_BRONZE_ACTIVITY)
)
query.awaitTermination()

In [0]:

import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="activity_bronze",
            batch_id=int(progress["batchId"]),
            input_rows=int(source.get("numInputRows", 0)),
            input_rows_per_second=float(source.get("inputRowsPerSecond", 0.0)),
            processed_rows_per_second=float(source.get("processedRowsPerSecond", 0.0)),
            processing_time_ms=int(
                progress.get("durationMs", {}).get("triggerExecution", 0)
            )
        )
    )

if metrics:
    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)
        

In [0]:
display(
    spark.sql(f"select * from {TABLE_BRONZE_ACTIVITY}")

)